In [1]:
"""
Dense LU vs Sparse LU vs Block Thomas Algorithm
Solves A x = b for block-tridiagonal matrices.

- Dense  : scipy.linalg.lu_factor / lu_solve   (LAPACK *getrf / *getrs)
- Sparse : scipy.sparse.linalg.splu            (SuperLU)
- Block Thomas : forward sweep + backward substitution  O(N * b^3)
"""

import time
import numpy as np
import scipy.linalg as sla
import scipy.sparse as sp
import scipy.sparse.linalg as spla


# ----------------------------------------------------------------------
# Test problem
# ----------------------------------------------------------------------
def make_block_tridiag(n_blocks, bs, seed=0):
    """Return (A_dense, A_sparse_csc, D, L, U) for a random diagonally-dominant
    block-tridiagonal matrix with n_blocks blocks of size bs x bs."""
    rng = np.random.default_rng(seed)
    n = n_blocks * bs

    D = [rng.standard_normal((bs, bs)) + bs * np.eye(bs) for _ in range(n_blocks)]
    L = [rng.standard_normal((bs, bs)) * 0.3 for _ in range(n_blocks - 1)]
    U = [rng.standard_normal((bs, bs)) * 0.3 for _ in range(n_blocks - 1)]

    A = np.zeros((n, n))
    for k in range(n_blocks):
        r = k * bs
        A[r:r+bs, r:r+bs] = D[k]
        if k < n_blocks - 1:
            A[r+bs:r+2*bs, r:r+bs] = L[k]
            A[r:r+bs, r+bs:r+2*bs] = U[k]

    A_sparse = sp.csc_matrix(A)
    return A, A_sparse, D, L, U


# ----------------------------------------------------------------------
# 1) Dense LU
# ----------------------------------------------------------------------
class DenseLU:
    """Factor once, solve many. Mirrors scipy.linalg.lu_factor/lu_solve."""
    def __init__(self, A_dense):
        self.lu_piv = sla.lu_factor(A_dense)

    def solve(self, b):
        return sla.lu_solve(self.lu_piv, b)

    def get_LUP(self, A_dense):
        P, L, U = sla.lu(A_dense)
        return P, L, U


# ----------------------------------------------------------------------
# 2) Sparse LU (SuperLU)
# ----------------------------------------------------------------------
class SparseLU:
    """Factor once, solve many. Wraps scipy.sparse.linalg.splu (SuperLU)."""
    def __init__(self, A_sparse_csc):
        self.lu = spla.splu(A_sparse_csc.tocsc())

    def solve(self, b):
        return self.lu.solve(b)


# ----------------------------------------------------------------------
# 3) Block Thomas Algorithm
# ----------------------------------------------------------------------
def block_thomas(D, L, U, b, dtype=None):
    N = len(D)
    if dtype is not None:
        D = [dk.astype(dtype) for dk in D]
        L = [lk.astype(dtype) for lk in L]
        U = [uk.astype(dtype) for uk in U]
        b = [bk.astype(dtype) for bk in b]
    else:
        D = [dk.copy() for dk in D]
        b = [bk.copy() for bk in b]

    for k in range(1, N):
        lu_piv = sla.lu_factor(D[k - 1])
        D[k] = D[k] - L[k - 1] @ sla.lu_solve(lu_piv, U[k - 1])
        b[k] = b[k] - L[k - 1] @ sla.lu_solve(lu_piv, b[k - 1])

    x = [None] * N
    x[N - 1] = sla.solve(D[N - 1], b[N - 1])
    for k in range(N - 2, -1, -1):
        lu_piv = sla.lu_factor(D[k])
        x[k] = sla.lu_solve(lu_piv, b[k] - U[k] @ x[k + 1])
    return x


def block_thomas_from_dense(A, b, block_size):
    """Extract blocks from dense matrix and solve with block_thomas."""
    n = A.shape[0]
    assert n % block_size == 0
    N = n // block_size
    bs = block_size

    D = [A[k*bs:(k+1)*bs, k*bs:(k+1)*bs] for k in range(N)]
    L = [A[(k+1)*bs:(k+2)*bs, k*bs:(k+1)*bs] for k in range(N - 1)]
    U = [A[k*bs:(k+1)*bs, (k+1)*bs:(k+2)*bs] for k in range(N - 1)]
    b_blocks = [b[k*bs:(k+1)*bs] for k in range(N)]

    return np.concatenate(block_thomas(D, L, U, b_blocks), axis=0)


# ----------------------------------------------------------------------
# Benchmark: all three solvers
# ----------------------------------------------------------------------
def bench(n_blocks, bs, n_rhs=1, seed=0):
    A, As, D, L, U = make_block_tridiag(n_blocks, bs, seed)
    n = A.shape[0]
    
    rng = np.random.default_rng(seed + 1)
    B = rng.standard_normal((n, n_rhs)) if n_rhs > 1 else rng.standard_normal(n)
    b_blocks = [B[k*bs:(k+1)*bs] for k in range(n_blocks)]

    # Dense LU
    t0 = time.perf_counter(); dlu = DenseLU(A);         t_dense_f = time.perf_counter() - t0
    t0 = time.perf_counter(); xd  = dlu.solve(B);       t_dense_s = time.perf_counter() - t0

    # Sparse LU
    t0 = time.perf_counter(); slu = SparseLU(As);       t_sparse_f = time.perf_counter() - t0
    t0 = time.perf_counter(); xs  = slu.solve(B);       t_sparse_s = time.perf_counter() - t0

    # Block Thomas
    t0 = time.perf_counter()
    xbt = np.concatenate(block_thomas(D, L, U, b_blocks, dtype=np.float64), axis=0)
    t_bt = time.perf_counter() - t0

    res_dense  = np.linalg.norm(A @ xd  - B) / np.linalg.norm(B)
    res_sparse = np.linalg.norm(A @ xs  - B) / np.linalg.norm(B)
    res_bt     = np.linalg.norm(A @ xbt - B) / np.linalg.norm(B)
    agree_ds   = np.linalg.norm(xd - xs)  / np.linalg.norm(xd)
    agree_dbt  = np.linalg.norm(xd - xbt) / np.linalg.norm(xd)
    nnz_frac   = As.nnz / (n * n)

    print(f"n={n:5d} ({n_blocks} blocks x {bs})  nnz={nnz_frac:6.2%} of dense")
    print(f"condition number={np.linalg.cond(A)}")
    print(f"  dense  LU    : factor {t_dense_f*1e3:8.2f} ms   solve {t_dense_s*1e3:7.3f} ms   residual {res_dense:.1e}")
    print(f"  sparse LU    : factor {t_sparse_f*1e3:8.2f} ms   solve {t_sparse_s*1e3:7.3f} ms   residual {res_sparse:.1e}")
    print(f"  block Thomas :         {t_bt*1e3:8.2f} ms                              residual {res_bt:.1e}")
    print(f"  dense vs sparse agreement : {agree_ds:.1e}")
    print(f"  dense vs block Thomas     : {agree_dbt:.1e}")
    print(f"  factor speedup sparse/BT : {(t_sparse_f+t_sparse_s)/t_bt:.1f}x")
    print()


if __name__ == "__main__":
    print("Dense LU vs Sparse LU vs Block Thomas\n")
    for n_blocks, bs in [(4, 100), (8, 100), (12, 100), (100, 64)]:
        bench(n_blocks, bs)

Dense LU vs Sparse LU vs Block Thomas

n=  400 (4 blocks x 100)  nnz=62.50% of dense
condition number=1.345360561729934
  dense  LU    : factor     7.68 ms   solve   0.158 ms   residual 1.0e-15
  sparse LU    : factor     6.38 ms   solve   0.077 ms   residual 8.4e-16
  block Thomas :             5.59 ms                              residual 7.6e-16
  dense vs sparse agreement : 9.0e-16
  dense vs block Thomas     : 7.8e-16
  factor speedup sparse/BT : 1.2x

n=  800 (8 blocks x 100)  nnz=34.38% of dense
condition number=1.3589165907744611
  dense  LU    : factor    62.60 ms   solve   0.347 ms   residual 1.1e-15
  sparse LU    : factor    12.87 ms   solve   0.162 ms   residual 8.5e-16
  block Thomas :            12.80 ms                              residual 7.7e-16
  dense vs sparse agreement : 8.3e-16
  dense vs block Thomas     : 8.6e-16
  factor speedup sparse/BT : 1.0x

n= 1200 (12 blocks x 100)  nnz=23.61% of dense
condition number=1.356236600166932
  dense  LU    : factor   207.22